# Propagators and sampling

Propagation is the numerical heart of physicaloptix. A propagator carries a
field from one plane to the next by evaluating a diffraction integral. The
far-field (focal-plane) case is a Fourier transform,

$$E(\mathbf{x}) = \iint A(\mathbf{u})\,e^{i\phi(\mathbf{u})}\,
e^{-2\pi i\,\mathbf{x}\cdot\mathbf{u}}\; d^2\mathbf{u},$$

written as a matrix discrete Fourier transform with continuous-transform
weights. This notebook covers what that buys you (focal sampling you choose
freely), where it bites (aliasing, caught at construction), the near-field
{class}`~physicaloptix.Fresnel` propagator, and the fact that a gradient flows
through all of it.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

from physicaloptix import (
    Field,
    Fraunhofer,
    Fresnel,
    Grid,
    OpticalPath,
    PlaneKind,
    Stage,
    mft_sampling_parameter,
)

WL, DIAM = 500.0, 6.0  # wavelength (nm), aperture diameter (m)
pupil = Grid.pupil(96)
u = np.asarray(pupil.coords)
ug, vg = np.meshgrid(u, u)
aperture = (np.hypot(ug, vg) <= 0.5).astype(complex)
flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)

## Choose your focal sampling

Unlike an FFT, the matrix transform decouples the output sampling from the
input array size. The focal pitch $\Delta x$ and pixel count $N_\text{out}$ are
yours to set independently of the 96-by-96 pupil: the field of view is
$N_\text{out}\,\Delta x$ and the resolution is $\Delta x$. The same pupil below
goes to a wide coarse grid, a medium grid, and a finely oversampled core, with
no zero-padding and no change to the pupil.

In [ ]:
samplings = [
    ("wide, coarse", Grid.focal(96, 0.5)),
    ("medium", Grid.focal(160, 0.15)),
    ("oversampled core", Grid.focal(160, 0.03)),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (title, grid) in zip(axes, samplings):
    out, _ = OpticalPath(
        stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=grid)),)
    ).propagate(flat)
    img = np.abs(np.asarray(out.data)) ** 2
    e = grid.extent
    im = ax.imshow(
        img / img.max(),
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-6, 1),
        extent=[-e, e, -e, e],
        interpolation="none",
    )
    ax.set_title(f"{title}\n{grid.dx:g} lambda/D per pixel, FOV +-{e:.1f}")
    ax.set_xlabel(r"$\lambda/D$")
plt.show()

## The sampling gate

Freedom to choose the sampling means you can also choose it badly. A focal grid
too wide for the pupil resolution aliases: light that belongs outside the field
of view wraps back in. The adequacy is a Nyquist ratio

$$p = \min(p_\text{in}, p_\text{out}), \qquad p \ge 1 \ \text{well sampled},\quad
p < 1 \ \text{aliasing},$$

which {func}`~physicaloptix.mft_sampling_parameter` returns from the two
coordinate axes. `Fraunhofer` computes the same number at construction and, by
its `on_undersampled` policy, will raise, warn, or silently record it. The
curve below crosses $p = 1$ as the pixels grow; past it the Airy rings fold back
on themselves.

In [ ]:
scales = np.linspace(0.1, 2.0, 40)
ps = np.array(
    [
        float(mft_sampling_parameter(pupil.coords, Grid.focal(96, s).coords))
        for s in scales
    ]
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
axes[0].plot(scales, ps, color="C0")
axes[0].axhline(1.0, color="C3", ls="--", lw=1)
axes[0].fill_between(scales, 0, 1, color="C3", alpha=0.12)
axes[0].text(1.4, 0.4, "aliasing", color="C3")
axes[0].set_xlabel("focal pixel scale [lambda/D]")
axes[0].set_ylabel("sampling parameter p")
axes[0].set_title("sampling gate")

for ax, s in ((axes[1], 0.2), (axes[2], 1.5)):
    grid = Grid.focal(96, s)
    prop = Fraunhofer(grid_in=pupil, grid_out=grid, on_undersampled="record")
    out, _ = OpticalPath(stages=(Stage("sci", prop),)).propagate(flat)
    img = np.abs(np.asarray(out.data)) ** 2
    e = grid.extent
    ax.imshow(
        img / img.max(),
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-5, 1),
        extent=[-e, e, -e, e],
        interpolation="none",
    )
    tag = "well sampled" if prop.sampling_parameter >= 1 else "undersampled"
    ax.set_title(f"{s} lambda/D: p = {prop.sampling_parameter:.2f} ({tag})")
    ax.set_xlabel(r"$\lambda/D$")
plt.show()

## Near-field diffraction: Fresnel

`Fraunhofer` is the far-field limit. Over a finite distance $z$ the field obeys
Fresnel diffraction, governed by the dimensionless

$$\alpha = \frac{z\,\lambda}{D^2},$$

proportional to the reciprocal of the Fresnel number (which conventionally uses the aperture radius). At $\alpha = 0$ the beam is the sharp-edged
aperture; propagate a short way and the edges develop diffraction fringes; go
further and the whole beam reorganizes. `Fresnel` stays on the same grid and
needs the physical beam diameter and wavelength to form $\alpha$. These near-field panels deliberately run below the conservative sampling gate ($p < 1$), with the propagators set to record rather than raise: the gate is a worst-case bound, and the single-frequency, band-limited fields used here stay accurate past it.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, alpha in zip(axes, (0.0, 0.004, 0.02, 0.08)):
    if alpha == 0.0:
        img = np.abs(np.asarray(flat.data)) ** 2
    else:
        z = alpha * DIAM**2 / (WL * 1e-9)
        fr = Fresnel(
            grid=pupil,
            distance_m=z,
            beam_diameter_m=DIAM,
            wavelength_nm=WL,
            plane_in=PlaneKind.PUPIL,
            plane_out=PlaneKind.INTERMEDIATE,
            on_undersampled="record",
        )
        out, _ = OpticalPath(stages=(Stage("z", fr),)).propagate(flat)
        img = np.abs(np.asarray(out.data)) ** 2
    ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(f"alpha = {alpha:g}")
    ax.set_xticks([]), ax.set_yticks([])
fig.suptitle("near-field intensity: edge fringes grow with propagation distance")
plt.show()

### Talbot: phase becomes amplitude

Fresnel propagation reveals something the far field hides. A pure phase ripple
of frequency $k$ cycles across the aperture, carried a distance $\alpha$,
acquires an amplitude ripple with weight

$$\left|\sin(\pi\,\alpha\,k^2)\right|,$$

the Talbot effect. This phase-to-amplitude conversion is the physical lever a
second, out-of-pupil deformable mirror uses to control the amplitude of a
wavefront. The measured modulation depth tracks the analytic curve.

In [ ]:
k = 4.0
a_rad = 0.02  # small phase amplitude, radians
phase = a_rad * np.cos(2 * np.pi * k * ug)
pure_phase = Field(
    data=jnp.asarray(aperture * np.exp(1j * phase)), grid=pupil, plane=PlaneKind.PUPIL
)
inner = (ug**2 + vg**2) <= 0.16
cosk = np.cos(2 * np.pi * k * ug)

alphas = np.linspace(0.005, 0.06, 12)
depth = []
for alpha in alphas:
    z = alpha * DIAM**2 / (WL * 1e-9)
    fr = Fresnel(
        grid=pupil,
        distance_m=z,
        beam_diameter_m=DIAM,
        wavelength_nm=WL,
        plane_in=PlaneKind.PUPIL,
        plane_out=PlaneKind.INTERMEDIATE,
        on_undersampled="record",
    )
    path = OpticalPath(stages=(Stage("z", fr),))
    a_rip = np.abs(np.asarray(path.propagate(pure_phase)[0].data))
    a_ref = np.abs(np.asarray(path.propagate(flat)[0].data))
    delta = a_rip[inner] / a_ref[inner] - 1.0
    depth.append(abs(2 * np.mean(delta * cosk[inner])))

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(alphas, np.array(depth) / a_rad, "o-", label="measured amplitude / phase")
ax.plot(
    alphas,
    np.abs(np.sin(np.pi * alphas * k**2)),
    "--",
    label=r"$|\sin(\pi\,\alpha k^2)|$",
)
ax.set_xlabel(r"$\alpha = z\lambda/D^2$")
ax.set_ylabel("amplitude modulation / phase amplitude")
ax.set_title(f"Talbot phase-to-amplitude conversion, k = {k:.0f}")
ax.legend()
plt.show()

## Propagation is differentiable

A path is a pure JAX function, so a gradient flows through the whole
propagation with respect to anything that shapes the wavefront. Take the
on-axis peak intensity (a Strehl proxy) as a function of a defocus amplitude
$a$: one `jax.grad` call gives $\partial\,\text{peak}/\partial a$ through the
Fraunhofer transform. The gradient is the slope of the response curve, zero at
best focus and steepening as the core spreads. It is exactly this gradient a
wavefront-control loop follows to dig a dark hole.

In [ ]:
focal = Grid.focal(128, 0.1)
tele = OpticalPath(stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),))
defocus = jnp.asarray((2.0 * (ug**2 + vg**2) - 0.5) * aperture.real)


def peak_intensity(a):
    wavefront = jnp.asarray(aperture) * jnp.exp(1j * 2 * jnp.pi * a * defocus)
    out, _ = tele.propagate(Field(data=wavefront, grid=pupil, plane=PlaneKind.PUPIL))
    return out.intensity().max()


grad_peak = jax.grad(peak_intensity)
aa = np.linspace(-0.35, 0.35, 41)
vals = np.array([float(peak_intensity(a)) for a in aa])

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(aa, vals, color="C0", label="peak intensity")
for a0 in (-0.2, 0.15):
    v0, g0 = float(peak_intensity(a0)), float(grad_peak(a0))
    ax.plot([a0 - 0.08, a0 + 0.08], [v0 - 0.08 * g0, v0 + 0.08 * g0], color="C3", lw=2)
    ax.plot(a0, v0, "o", color="C3")
ax.set_xlabel("defocus amplitude a [waves]")
ax.set_ylabel("peak intensity (Strehl proxy)")
ax.set_title("gradients (red) run through the propagation")
ax.legend()
plt.show()

## Where to go next

- **[Building a coronagraph](03_Building_a_Coronagraph)** folds a focal-plane
  mask and a Lyot stop into a path and wraps it as a reusable coronagraph.
- The [architecture page](../explanation/architecture) covers the
  continuous-Fourier transform and the plane-tag model in more depth.
- The gradient shown here is what a wavefront-control library like `tiptilt`
  follows to correct aberrations.